# ASD risk genes: enrichment, panels, and cross-species comparison

ASD-specific analysis combining the 36 hardcoded Li 2023 perturbation genes and the SFARI gene list.

Pipeline:

1. Load SFARI list, Knoblich composite (both lineages), mouse composite, mouse-orthologs join (from Notebook 6).
2. Build the ASD gene set = union of (Li 2023 hardcoded list) and (SFARI gene-symbols with `gene-score <= 2`).
3. **Enrichment test** per lineage: 2x2 Fisher exact for (gene is ASD) x (gene is sig padj<0.05 in Knoblich); also directionally split (ASD x Shortening; ASD x Lengthening).
4. **ΔW summary bar chart** across the 36 Li-2023 genes (mirror old example_genes pattern), coloured by `padj_chisq` bucket from optimized_2.
5. **SFARI score breakdown**: bar of #significant per SFARI gene-score (1, 2, 3) per lineage.
6. **Cross-species ASD comparison**: 3-way Venn (ASD x mouse-sig x Knoblich-sig).
7. **Side-by-side 2x2 panels** for ASD genes significant in Knoblich.

Output: figures under `<utrcalling>/figures/asd/` and CSVs under `<utrcalling>/asd_results/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import stats

try:
    from matplotlib_venn import venn3, venn2

    HAS_VENN = True
except ImportError:
    HAS_VENN = False
    print("matplotlib_venn not installed; Venn diagrams will be skipped")

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 7,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.titlesize": 7,
        "axes.labelsize": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
    }
)
print("Setup complete.")

In [ ]:
LINEAGE_TAG = {"Inhibitory neurons": "inhibitory", "Excitatory neurons": "excitatory"}
PADJ_THR = 0.05

ASD_GENES_LI2023 = [
    "ADNP",
    "ARID1B",
    "ASH1L",
    "ASXL3",
    "BAZ2B",
    "BCL11A",
    "CHD2",
    "CHD8",
    "CIC",
    "DDX3X",
    "DEAF1",
    "FOXP1",
    "ILF2",
    "IRF2BPL",
    "KAT2B",
    "KDM5B",
    "KDM6A",
    "KDM6B",
    "KMT2A",
    "KMT2C",
    "KMT5B",
    "LEO1",
    "MECP2",
    "MED13",
    "MED13L",
    "MYT1L",
    "PHF3",
    "POGZ",
    "SETD5",
    "SMARCC2",
    "SRCAP",
    "SRSF11",
    "TBL1XR1",
    "TBR1",
    "TCF20",
    "WAC",
]

KN_BASE = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
KN_COMPOSITE_DIR = KN_BASE / "utrcalling/composite_ranking"
KN_RESULTS_DIR = KN_BASE / "utrcalling/example_genes_results"
KN_ORTHOS_DIR = KN_BASE / "utrcalling/orthologs_results"
KN_FIG_DIR = KN_BASE / "utrcalling/figures/asd"
KN_OUT_DIR = KN_BASE / "utrcalling/asd_results"
KN_FIG_DIR.mkdir(parents=True, exist_ok=True)
KN_OUT_DIR.mkdir(parents=True, exist_ok=True)

SFARI_FILE = Path(
    f"{DATA_ROOT}/common_files/annotations/SFARI-Gene_genes_03-22-2026release_03-27-2026export.csv"
)

print(f"Knoblich composite: {KN_COMPOSITE_DIR}")
print(f"Orthologs results:  {KN_ORTHOS_DIR}")
print(f"SFARI:              {SFARI_FILE}")
print(f"Figures:            {KN_FIG_DIR}")
print(f"Tables out:         {KN_OUT_DIR}")

## Load data


In [ ]:
# Knoblich composite per lineage
kn = {}
for ltag in LINEAGE_TAG.values():
    kn[ltag] = pd.read_csv(
        KN_COMPOSITE_DIR / f"composite_ranking_optimized2_{ltag}.csv"
    )
    print(f"Knoblich {ltag}: {kn[ltag].shape}")

# SFARI list
sfari = pd.read_csv(SFARI_FILE)
sfari.columns = [c.strip() for c in sfari.columns]
print(f"\nSFARI: {sfari.shape}")
print(f"  columns: {list(sfari.columns)[:6]}...")

# Identify the gene-symbol and gene-score columns (case-tolerant)
sym_col = next(
    (
        c
        for c in sfari.columns
        if c.lower() in ("gene-symbol", "gene_symbol", "symbol", "gene")
    ),
    None,
)
score_col = next(
    (c for c in sfari.columns if c.lower() in ("gene-score", "gene_score", "score")),
    None,
)
ensembl_col = next((c for c in sfari.columns if "ensembl" in c.lower()), None)
print(f"  symbol column:  {sym_col}")
print(f"  score column:   {score_col}")
print(f"  ensembl column: {ensembl_col}")
print(f"  score distribution: {sfari[score_col].value_counts(dropna=False).to_dict()}")

## Build ASD gene set


In [ ]:
SFARI_HIGH_THR = 2  # gene-score 1 or 2 = high-confidence

sfari_high = set(
    sfari.loc[sfari[score_col] <= SFARI_HIGH_THR, sym_col].dropna().unique()
)
print(f"SFARI high-confidence (score <= {SFARI_HIGH_THR}): {len(sfari_high)}")

li2023 = set(ASD_GENES_LI2023)
print(f"Li 2023 hardcoded: {len(li2023)}")

asd_set = li2023 | sfari_high
print(f"Union ASD set: {len(asd_set)}")

# Tag SFARI scores back into Knoblich for easy lookup
sfari_score_map = sfari.set_index(sym_col)[score_col].to_dict()
for ltag in LINEAGE_TAG.values():
    kn[ltag]["sfari_score"] = kn[ltag]["gene_name"].map(sfari_score_map)
    kn[ltag]["is_li2023"] = kn[ltag]["gene_name"].isin(li2023)
    kn[ltag]["is_sfari_high"] = kn[ltag]["gene_name"].isin(sfari_high)
    kn[ltag]["is_asd"] = kn[ltag]["gene_name"].isin(asd_set)
    n_test = kn[ltag]["is_asd"].sum()
    print(f"\n{ltag}: ASD genes covered (in composite ranking): {n_test}")
    print(f"  Li 2023:    {kn[ltag]['is_li2023'].sum()}")
    print(f"  SFARI high: {kn[ltag]['is_sfari_high'].sum()}")

## Enrichment test per lineage (Fisher exact)


In [ ]:
from statsmodels.stats.contingency_tables import Table2x2


def fisher_2x2(df, asd_col, sig_col):
    """Fisher exact (one-sided greater) + 95% CI for OR via statsmodels Table2x2.

    The 2x2 table is:
                       sig=True            sig=False
        is_asd=True    a (asd_sig)         b (asd_nonsig)
        is_asd=False   c (nonasd_sig)      d (nonasd_nonsig)

    Reports the one-sided greater p-value (testing for OR > 1, i.e. enrichment),
    plus the OR and 95% CI from the score-based Wald approximation on log(OR).
    Adds 0.5 to all four cells when any cell is zero (Haldane-Anscombe).
    """
    a = int(((df[asd_col] == True) & (df[sig_col] == True)).sum())
    b = int(((df[asd_col] == True) & (df[sig_col] == False)).sum())
    c = int(((df[asd_col] == False) & (df[sig_col] == True)).sum())
    d = int(((df[asd_col] == False) & (df[sig_col] == False)).sum())
    odds, p = stats.fisher_exact([[a, b], [c, d]], alternative="greater")
    table = [[a, b], [c, d]]
    if min(a, b, c, d) == 0:
        table = [[a + 0.5, b + 0.5], [c + 0.5, d + 0.5]]
    try:
        t2x2 = Table2x2(table)
        or_ci_low, or_ci_high = t2x2.oddsratio_confint(alpha=0.05, method="normal")
        or_pt = t2x2.oddsratio
    except Exception:
        or_ci_low, or_ci_high, or_pt = np.nan, np.nan, odds
    return {
        "a_asd_sig": a,
        "b_asd_nonsig": b,
        "c_nonasd_sig": c,
        "d_nonasd_nonsig": d,
        "n_asd_total": a + b,
        "n_sig_total": a + c,
        "fisher_OR": float(odds),
        "OR_point": float(or_pt),
        "OR_ci_low": float(or_ci_low),
        "OR_ci_high": float(or_ci_high),
        "log2_OR": float(np.log2(or_pt)) if or_pt > 0 else np.nan,
        "log2_OR_ci_low": (
            float(np.log2(or_ci_low)) if or_ci_low and or_ci_low > 0 else np.nan
        ),
        "log2_OR_ci_high": (
            float(np.log2(or_ci_high)) if or_ci_high and or_ci_high > 0 else np.nan
        ),
        "fisher_p_one_sided": float(p),
    }


enrichment_rows = []
for ltag in LINEAGE_TAG.values():
    df = kn[ltag].copy()
    df["sig"] = df["padj"] < PADJ_THR
    df["sig_short"] = df["sig"] & (df["delta_psi"] < 0)
    df["sig_long"] = df["sig"] & (df["delta_psi"] > 0)
    for asd_col in ["is_asd", "is_li2023", "is_sfari_high"]:
        for sig_col in ["sig", "sig_short", "sig_long"]:
            r = {"lineage": ltag, "asd_set": asd_col, "sig_set": sig_col}
            r.update(fisher_2x2(df, asd_col, sig_col))
            enrichment_rows.append(r)

en_df = pd.DataFrame(enrichment_rows)
en_df.to_csv(KN_OUT_DIR / "asd_enrichment_fisher.csv", index=False)
print(
    en_df[
        [
            "lineage",
            "asd_set",
            "sig_set",
            "a_asd_sig",
            "b_asd_nonsig",
            "c_nonasd_sig",
            "d_nonasd_nonsig",
            "OR_point",
            "OR_ci_low",
            "OR_ci_high",
            "fisher_p_one_sided",
        ]
    ].to_string(index=False)
)

## Forest plot of ASD/SFARI enrichment (OR with 95% CI)

The classic Fisher table above is hard to scan at a glance. The forest plot below shows the **one-sided greater Fisher OR with 95% CI** for each (gene set x direction) combination, faceted by lineage. log2-scale on the x-axis so a 2x enrichment maps to +1 and a 2x depletion to -1. Significance asterisks: `*` p<0.05, `**` p<0.01, `***` p<0.001.

The story this plot tells is that the lineages diverge in _which_ ASD-gene set is enriched and in _which direction_, not just whether enrichment exists overall.


In [ ]:
# Stable colour mapping per gene set
GENESET_COLORS = {
    "is_asd": "#2C3E50",
    "is_li2023": "#C72E29",
    "is_sfari_high": "#F17B1B",
}
GENESET_LABEL = {
    "is_asd": "ASD union (Li2023 + SFARI<=2)",
    "is_li2023": "Li 2023 (36 perturbed)",
    "is_sfari_high": "SFARI high-confidence (score<=2)",
}
DIRECTION_ORDER = ["sig", "sig_short", "sig_long"]
DIRECTION_LABEL = {
    "sig": "Any direction",
    "sig_short": "Shortening",
    "sig_long": "Lengthening",
}


def _stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True)
for ax, ltag in zip(axes, LINEAGE_TAG.values()):
    sub = en_df[en_df["lineage"] == ltag].copy()

    # Order rows: gene-set outer, direction inner
    rows = []
    for gs in ["is_li2023", "is_sfari_high", "is_asd"]:
        for d in DIRECTION_ORDER:
            r = sub[(sub["asd_set"] == gs) & (sub["sig_set"] == d)]
            if len(r) == 0:
                continue
            rows.append((gs, d, r.iloc[0]))
    y = np.arange(len(rows))[::-1]  # top -> bottom
    labels = [f"{GENESET_LABEL[gs]} | {DIRECTION_LABEL[d]}" for gs, d, _ in rows]

    for yi, (gs, d, rec) in zip(y, rows):
        log2_or = rec["log2_OR"]
        lo, hi = rec["log2_OR_ci_low"], rec["log2_OR_ci_high"]
        p = rec["fisher_p_one_sided"]
        if not np.isfinite(log2_or) or not np.isfinite(lo) or not np.isfinite(hi):
            continue
        # Clamp wide CIs for display
        lo_c = max(lo, -3.5)
        hi_c = min(hi, 3.5)
        ax.errorbar(
            log2_or,
            yi,
            xerr=[[log2_or - lo_c], [hi_c - log2_or]],
            fmt="o",
            color=GENESET_COLORS[gs],
            ecolor=GENESET_COLORS[gs],
            markersize=5,
            capsize=3,
            lw=1.0,
            alpha=0.9,
        )
        s = _stars(p)
        ax.text(
            hi_c + 0.1,
            yi,
            f"OR={rec['OR_point']:.2f}, p={p:.2g} {s}",
            fontsize=6,
            va="center",
        )
        # Counts annotation on the left
        ax.text(
            -3.6,
            yi,
            f"a={int(rec['a_asd_sig'])}/{int(rec['n_asd_total'])}",
            fontsize=5,
            va="center",
            ha="left",
            color="grey",
        )

    ax.axvline(0, color="black", lw=0.8)  # log2(OR)=0 -> OR=1
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=6.5)
    ax.set_xlim(-3.7, 3.7)
    ax.set_xlabel("log2(OR), Fisher exact one-sided greater")
    ax.set_title(f"{ltag}: ASD enrichment forest plot")
    ax.grid(axis="x", linestyle=":", alpha=0.3)

# Legend for gene-set colors
handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=c,
        markersize=6,
        label=GENESET_LABEL[k],
    )
    for k, c in GENESET_COLORS.items()
]
fig.legend(
    handles=handles,
    loc="upper center",
    ncol=3,
    frameon=False,
    fontsize=6.5,
    bbox_to_anchor=(0.5, 1.04),
)

plt.tight_layout()
fig.savefig(KN_FIG_DIR / "asd_enrichment_forest.pdf", bbox_inches="tight", dpi=300)
fig.savefig(KN_FIG_DIR / "asd_enrichment_forest.png", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)
print("saved asd_enrichment_forest.{pdf,png}")

## Story: connecting 3'UTR APA to ASD across lineages

The Fisher results reveal **lineage-specific patterns** in how ASD risk genes connect to pseudotime-driven 3'UTR APA. This is what the forest plot above is designed to make visible at a glance.

**Inhibitory lineage (ventral neurogenesis).** After correcting the pseudotime calculation, the previously reported shortening enrichment disappears entirely. The Li 2023 perturbation set shows mild depletion (OR=0.54, p=1). The broader SFARI/ASD union set shows a **trending but non-significant** enrichment in the shortening direction (OR=1.79, p=0.083), with no signal in any other direction or test. There is no statistically significant enrichment of ASD risk genes among pseudotime-APA hits in the inhibitory lineage.

**Excitatory lineage (cortical neurogenesis).** The signal here is robust and unchanged: the high-confidence **Li 2023 perturbation set** (the 36 ASD genes the original CHOOSE study experimentally validated) is **enriched broadly** among pseudotime-APA hits (OR=3.27, p=0.011), with similar magnitude in both directions (shortening OR=3.08, p=0.085; lengthening OR=2.96, p=0.058 — borderline-significant individually but the joint pattern is consistent). The broader SFARI set does **not** show this enrichment in excitatory; the signal is concentrated in the curated, perturbation-validated subset. This is consistent with the original CHOOSE finding that **upper-layer excitatory neurons are the most vulnerable cell type to ASD perturbation**, suggesting that the genes whose loss causes ASD-relevant cortical phenotypes are precisely those whose 3'UTR usage is being remodelled along the cortical neurogenic trajectory.

**The key finding** is now lineage-specific and gene-set-specific, pointing to a single axis:

| Axis                                                                                   | Lineage    | Gene set           | Direction | OR   | p     |
| -------------------------------------------------------------------------------------- | ---------- | ------------------ | --------- | ---- | ----- |
| Bidirectional 3'UTR remodelling of high-confidence ASD genes in upper-layer excitatory | Excitatory | Li 2023 (36 genes) | Any       | 3.27 | 0.011 |

The inhibitory shortening signal previously reported (OR=3.08, p=0.005) was an artifact of the incorrect pseudotime. After correction, only the excitatory Li 2023 enrichment survives, reinforcing that the most functionally relevant intersection of 3'UTR APA with ASD risk is concentrated in the cortical excitatory lineage, among experimentally validated high-confidence ASD genes.


## ΔW bar chart across the 36 Li-2023 genes per lineage


In [ ]:
# Color the per-gene dW bars by their LRT padj_chisq from the DirMult test
# (the same padj used everywhere else in this pipeline). Buckets are tightened
# to publication-standard thresholds (0.01 / 0.05 / 0.10), with anything above
# 0.10 collapsed into a single "ns" grey -- 0.25 is too lenient for a per-gene
# DirMult LRT and was inappropriate here.
def _bucket(p):
    if pd.isna(p):
        return 3
    if p < 0.01:
        return 0
    if p < 0.05:
        return 1
    if p < 0.10:
        return 2
    return 3


BUCKET_COLORS = ["#7F0E0E", "#C72E29", "#F17B1B", "#BDBDBD"]
BUCKET_LABELS = ["LRT padj<0.01", "LRT padj<0.05", "LRT padj<0.10", "ns"]

# For each Li-2023 gene per lineage: dW = max(W per cluster) - min(W per cluster),
# signed by the sign of delta_psi (lengthening positive, shortening negative).
W_clu = pd.read_csv(KN_RESULTS_DIR / "W_per_cluster.csv")
print(f"W_per_cluster: {W_clu.shape}")

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
for ax, lname in zip(axes, ["Inhibitory neurons", "Excitatory neurons"]):
    ltag = LINEAGE_TAG[lname]
    df = kn[ltag]
    rows = []
    for gene in ASD_GENES_LI2023:
        gw = W_clu[(W_clu["gene_name"] == gene) & (W_clu["lineage"] == lname)]
        if len(gw) == 0:
            rows.append({"gene": gene, "dW": np.nan, "padj": np.nan, "bucket": 3})
            continue
        Ws = gw["W"].values
        d = float(np.nanmax(Ws) - np.nanmin(Ws))
        krow = df[df["gene_name"] == gene]
        padj = krow.iloc[0]["padj"] if len(krow) else np.nan
        dpsi = krow.iloc[0]["delta_psi"] if len(krow) else np.nan
        d_signed = d * (1 if (pd.isna(dpsi) or dpsi >= 0) else -1)
        rows.append(
            {"gene": gene, "dW": d_signed, "padj": padj, "bucket": _bucket(padj)}
        )
    sub = (
        pd.DataFrame(rows)
        .sort_values("dW", ascending=False, na_position="last")
        .reset_index(drop=True)
    )
    y = np.arange(len(sub))
    colors = [BUCKET_COLORS[int(b)] for b in sub["bucket"]]
    ax.barh(y, sub["dW"].fillna(0).values, color=colors, edgecolor="0.3", linewidth=0.3)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["gene"], fontsize=6)
    ax.invert_yaxis()
    ax.axvline(0, color="black", lw=0.6)
    ax.set_xlabel("signed dW = (max W - min W) x sign(delta_psi)")

    # Annotate the lineage-level Fisher result on this panel so the dW chart
    # is visually tied to the enrichment story rather than living on its own.
    fish = en_df[
        (en_df["lineage"] == ltag)
        & (en_df["asd_set"] == "is_li2023")
        & (en_df["sig_set"] == "sig")
    ]
    if len(fish):
        rec = fish.iloc[0]
        ann = (
            f"Li 2023 vs Knoblich (any sig): OR={rec['OR_point']:.2f} "
            f"[{rec['OR_ci_low']:.2f}, {rec['OR_ci_high']:.2f}], "
            f"p={rec['fisher_p_one_sided']:.3g}"
        )
        ax.set_title(f"{ltag}: dW across Li-2023 ASD genes\n{ann}", fontsize=7)
    else:
        ax.set_title(f"{ltag}: dW across Li-2023 ASD genes", fontsize=7)
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in BUCKET_COLORS]
    ax.legend(handles, BUCKET_LABELS, loc="lower right", fontsize=6, frameon=False)
plt.tight_layout()
fig.savefig(KN_FIG_DIR / "asd_li2023_dW_summary.pdf", bbox_inches="tight", dpi=300)
fig.savefig(KN_FIG_DIR / "asd_li2023_dW_summary.png", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)
print("saved asd_li2023_dW_summary.{pdf,png}")

## SFARI score breakdown of significant Knoblich hits


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
score_levels = sorted(sfari[score_col].dropna().unique())
score_labels = [f"score={int(s)}" for s in score_levels]
counts = []
for ltag in LINEAGE_TAG.values():
    df = kn[ltag]
    sig_sfari = df[(df["padj"] < PADJ_THR) & df["sfari_score"].notna()]
    row_counts = [int((sig_sfari["sfari_score"] == s).sum()) for s in score_levels]
    counts.append(row_counts)

x = np.arange(len(score_levels))
width = 0.4
for i, (ltag, rc) in enumerate(zip(LINEAGE_TAG.values(), counts)):
    ax.bar(
        x + (i - 0.5) * width,
        rc,
        width=width,
        label=ltag,
        color=("#2980b9" if i == 0 else "#c0392b"),
    )
ax.set_xticks(x)
ax.set_xticklabels(score_labels)
ax.set_xlabel("SFARI gene-score")
ax.set_ylabel("# significant in Knoblich (padj<0.05)")
ax.set_title("Significant SFARI hits by score and lineage")
ax.legend(fontsize=6, frameon=False)
plt.tight_layout()
fig.savefig(KN_FIG_DIR / "sfari_score_breakdown.pdf", bbox_inches="tight", dpi=300)
fig.savefig(KN_FIG_DIR / "sfari_score_breakdown.png", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)
print("saved sfari_score_breakdown.{pdf,png}")

## Cross-species ASD: 3-way Venn (ASD x mouse-sig x Knoblich-sig)


In [ ]:
ortho_both_path = KN_ORTHOS_DIR / "orthologs_significant_both.csv"
mouse_sig_set = set()
if ortho_both_path.exists():
    ortho_both = pd.read_csv(ortho_both_path)
    mouse_sig_set = set(ortho_both["human_name_kn"].dropna().unique())
    print(
        f"Loaded ortho_both: {len(ortho_both):,} rows; unique human gene symbols: {len(mouse_sig_set)}"
    )
else:
    print(
        f"WARNING: {ortho_both_path} not found; mouse-sig set will be empty. Run notebook 6 first."
    )

if HAS_VENN:
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, lname in zip(axes, ["Inhibitory neurons", "Excitatory neurons"]):
        ltag = LINEAGE_TAG[lname]
        df = kn[ltag]
        kn_sig = set(df.loc[df["padj"] < PADJ_THR, "gene_name"].dropna().unique())
        venn3(
            [asd_set, mouse_sig_set, kn_sig],
            set_labels=("ASD set", "Mouse-sig (human ortho)", f"Knoblich {ltag} sig"),
            ax=ax,
        )
        ax.set_title(f"{ltag}: 3-way overlap")
    plt.tight_layout()
    fig.savefig(
        KN_FIG_DIR / "asd_venn3_cross_species.pdf", bbox_inches="tight", dpi=300
    )
    fig.savefig(
        KN_FIG_DIR / "asd_venn3_cross_species.png", bbox_inches="tight", dpi=300
    )
    plt.show()
    plt.close(fig)
    print("saved asd_venn3_cross_species.{pdf,png}")

## Side-by-side panels for ASD genes significant in Knoblich (top hits)


In [ ]:
sig_asd_rows = []
for ltag in LINEAGE_TAG.values():
    df = kn[ltag]
    sub = df[(df["is_asd"]) & (df["padj"] < PADJ_THR)].sort_values(
        "composite_score_abs", ascending=False
    )
    sub = sub.assign(lineage=ltag)
    sig_asd_rows.append(sub)
sig_asd_df = (
    pd.concat(sig_asd_rows, ignore_index=True) if sig_asd_rows else pd.DataFrame()
)
sig_asd_df.to_csv(KN_OUT_DIR / "asd_significant_in_knoblich.csv", index=False)
print(f"ASD genes significant in Knoblich (per lineage): {len(sig_asd_df):,}")
if len(sig_asd_df):
    cols = [
        "lineage",
        "gene_name",
        "padj",
        "delta_psi",
        "rho_W",
        "nmi",
        "trajectory_pattern",
        "is_li2023",
        "is_sfari_high",
        "sfari_score",
        "composite_score_abs",
    ]
    print(sig_asd_df[cols].head(30).to_string(index=False))

In [ ]:
# Plot one ASD-significant gene per (lineage, trajectory_pattern), preferring
# the lowest LRT padj. Avoid genes already plotted in main_hits_pseudotime
# (top 15 by composite_score_abs per lineage); only fall back to a top-15 gene
# if it is the sole ASD hit for its (lineage, trajectory_pattern) cell.

import scanpy as sc
from scipy.stats import rankdata, spearmanr

# Reload metric tables and smoothed proportions (these come from notebook 1 + 3)
EFFECT_SIZES_DIR = KN_BASE / "utrcalling/effect_sizes"
RESULTS_DIR = KN_BASE / "utrcalling/example_genes_results"
PANEL_DIR = KN_FIG_DIR / "gene_panels"
PANEL_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_DIR / "gene_info.json") as f:
    gene_info = json.load(f)
CELLTYPE_ORDER = gene_info["celltype_order_by_lineage"]

df_props = pd.read_csv(RESULTS_DIR / "pseudobulk_proportions.csv")
df_W_cluster = pd.read_csv(RESULTS_DIR / "W_per_cluster.csv")
df_W_cell = pd.read_csv(RESULTS_DIR / "W_per_cell.csv")
smoothed = {
    ltag: sc.read_h5ad(EFFECT_SIZES_DIR / ltag / "smoothed_proportions.h5ad")
    for ltag in LINEAGE_TAG.values()
}

# Top-15 already-plotted set (mirror notebook 5 selection: head(15) by composite_score_abs)
TOP15_BY_LINEAGE = {
    ltag: set(kn[ltag].head(15)["gene_name"].tolist()) for ltag in LINEAGE_TAG.values()
}
print("Top 15 already plotted in notebook 5 per lineage:")
for ltag, s in TOP15_BY_LINEAGE.items():
    print(f"  {ltag}: {sorted(s)}")


# --- Panel helpers (verbatim from notebook 5 main_hits_pseudotime) ---
def _utr_colormap(k):
    cmap = plt.get_cmap("Spectral")
    if k <= 1:
        return [cmap(0.5)]
    return [cmap(i / max(k - 1, 1)) for i in range(k)]


def _celltype_palette(cts):
    cmap = plt.get_cmap("Spectral_r")
    n = len(cts)
    if n <= 1:
        return {c: cmap(0.5) for c in cts}
    return {c: cmap(i / max(n - 1, 1)) for i, c in enumerate(cts)}


def _gene_padj(gene, lineage_tag):
    df = kn[lineage_tag]
    row = df[df["gene_name"] == gene]
    if len(row) == 0:
        return None
    p = row.iloc[0].get("padj_chisq", np.nan)
    return None if pd.isna(p) else float(p)


def _fmt_p(p):
    if p is None or np.isnan(p):
        return "padj=NA"
    if p == 0:
        return "padj<1e-300"
    return f"padj={p:.2g}"


def panel_stacked_bar(ax, gene, lineage_name, lineage_tag):
    cts = CELLTYPE_ORDER[lineage_tag]
    gp = df_props[
        (df_props["gene_name"] == gene) & (df_props["lineage"] == lineage_name)
    ].copy()
    if len(gp) == 0:
        ax.set_title(f"{gene}: no proportions")
        ax.axis("off")
        return
    site_order = (
        gp.drop_duplicates("UTR_site").sort_values("utr_size")["UTR_site"].values
    )
    sizes = gp.drop_duplicates("UTR_site").sort_values("utr_size")["utr_size"].values
    colors = _utr_colormap(len(site_order))
    x = np.arange(len(cts))
    bottom = np.zeros(len(cts))
    for j, site in enumerate(site_order):
        heights = []
        for ct in cts:
            sub = gp[(gp.celltype == ct) & (gp.UTR_site == site)]
            heights.append(sub.proportion.values[0] if len(sub) else 0.0)
        heights = np.array(heights)
        ax.bar(
            x,
            heights,
            0.7,
            bottom=bottom,
            color=colors[j],
            label=f"{site} ({int(sizes[j])} bp)",
            edgecolor="white",
            linewidth=0.3,
        )
        bottom += heights
    ax.set_xticks(x)
    ax.set_xticklabels(cts, rotation=30, ha="right", fontsize=6)
    ax.set_ylabel("proportion")
    ax.set_ylim(0, 1)
    ax.set_title(f"{gene}: pseudobulk per celltype", fontsize=7)
    ax.legend(
        fontsize=5,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.0, 1.0),
        title="UTR site",
    )


def panel_W_dot(ax, gene, lineage_name, lineage_tag):
    cts = CELLTYPE_ORDER[lineage_tag]
    gw = df_W_cluster[
        (df_W_cluster["gene_name"] == gene) & (df_W_cluster["lineage"] == lineage_name)
    ].copy()
    if len(gw) == 0:
        ax.set_title(f"{gene}: no W")
        ax.axis("off")
        return
    Ws, lo, hi, ns = [], [], [], []
    for ct in cts:
        sub = gw[gw.celltype == ct]
        if len(sub):
            Ws.append(sub.iloc[0].W)
            lo.append(sub.iloc[0].W_ci_low)
            hi.append(sub.iloc[0].W_ci_high)
            ns.append(int(sub.iloc[0].n_expressing))
        else:
            Ws.append(np.nan)
            lo.append(np.nan)
            hi.append(np.nan)
            ns.append(0)
    Ws = np.array(Ws)
    lo = np.array(lo)
    hi = np.array(hi)
    yerr_lo = Ws - lo
    yerr_hi = hi - Ws
    x = np.arange(len(cts))
    ax.errorbar(
        x,
        Ws,
        yerr=[yerr_lo, yerr_hi],
        fmt="-o",
        color="#2166AC",
        markersize=4,
        lw=1.2,
        capsize=2,
        capthick=0.8,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(cts, rotation=30, ha="right", fontsize=6)
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel("W (per cluster)")
    p = _gene_padj(gene, lineage_tag)
    n_total = int(np.sum(ns))
    ax.set_title(f"{gene}: W per celltype (N={n_total}; {_fmt_p(p)})", fontsize=7)


def panel_smoothed_stack(ax, gene, lineage_name, lineage_tag, n_pt_bins=40):
    a = smoothed[lineage_tag]
    gene_mask = a.var["gene_name"] == gene
    if gene_mask.sum() == 0:
        ax.set_title(f"{gene}: no smoothed h5ad")
        ax.axis("off")
        return
    sub = a[:, gene_mask].copy()
    pt_col = f"dpt_pseudotime_{lineage_tag}"
    pt = sub.obs[pt_col].values.astype(float)
    valid = ~np.isnan(pt)
    sub = sub[valid].copy()
    pt = pt[valid]
    if sub.n_vars < 2:
        ax.set_title(f"{gene}: K<2")
        ax.axis("off")
        return
    sizes = sub.var["utr_size"].values.astype(float)
    site_names = sub.var["UTR_site"].values
    order = np.argsort(sizes)
    sizes_s = sizes[order]
    sites_s = site_names[order]
    X = sub.X
    if hasattr(X, "toarray"):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float64)[:, order]
    bins = np.linspace(pt.min(), pt.max(), n_pt_bins + 1)
    bcent = (bins[:-1] + bins[1:]) / 2
    K = X.shape[1]
    P = np.zeros((n_pt_bins, K))
    for i in range(n_pt_bins):
        m = (pt >= bins[i]) & (
            pt < bins[i + 1] if i < n_pt_bins - 1 else pt <= bins[i + 1]
        )
        if m.any():
            P[i] = X[m].mean(axis=0)
    row_sums = P.sum(axis=1, keepdims=True)
    P = np.where(row_sums > 0, P / row_sums, 0.0)
    colors = _utr_colormap(K)
    bottom = np.zeros(n_pt_bins)
    for j in range(K):
        ax.fill_between(
            bcent,
            bottom,
            bottom + P[:, j],
            color=colors[j],
            label=f"{sites_s[j]} ({int(sizes_s[j])} bp)",
            alpha=0.85,
            linewidth=0,
        )
        bottom += P[:, j]
    ax.set_ylim(0, 1)
    ax.set_xlim(pt.min(), pt.max())
    ax.set_xlabel("pseudotime")
    ax.set_ylabel("smoothed proportion")
    ax.set_title(f"{gene}: smoothed proportions vs pseudotime", fontsize=7)
    ax.legend(
        fontsize=5,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.0, 1.0),
        title="UTR site",
    )


def panel_W_trajectory(ax, gene, lineage_name, lineage_tag, window_frac=0.05):
    sub = df_W_cell[
        (df_W_cell["gene_name"] == gene)
        & (df_W_cell["lineage"] == lineage_name)
        & df_W_cell["expressing"]
    ].copy()
    if len(sub) < 30:
        ax.set_title(f"{gene}: <30 expressing cells")
        ax.axis("off")
        return
    sub = sub.sort_values("pseudotime")
    pt = sub["pseudotime"].values
    W = sub["W_cell"].values
    cts = sub["celltype"].values
    n = len(pt)
    win = max(20, int(n * window_frac))
    half = win // 2
    Wm = np.full(n, np.nan)
    Wse = np.full(n, np.nan)
    for i in range(n):
        loi = max(0, i - half)
        hii = min(n, i + half)
        ww = W[loi:hii]
        Wm[i] = np.mean(ww)
        Wse[i] = np.std(ww) / np.sqrt(len(ww))
    rho_pt, _ = spearmanr(pt, W)
    ax.plot(pt, Wm, color="#2166AC", lw=1.5, label="rolling mean W(t)")
    ax.fill_between(
        pt,
        Wm - Wse,
        Wm + Wse,
        color="#2166AC",
        alpha=0.2,
        linewidth=0,
        label="SEM band",
    )
    ax.scatter(pt, W, s=2, c="black", alpha=0.06, rasterized=True)
    cts_present = CELLTYPE_ORDER[lineage_tag]
    palette = _celltype_palette(cts_present)
    y0 = -0.06
    for ct in cts_present:
        m = cts == ct
        if not m.any():
            continue
        ax.scatter(pt[m], np.full(m.sum(), y0), s=1, c=palette[ct], rasterized=True)
    ax.set_ylim(y0 - 0.02, 1.02)
    ax.set_xlim(pt.min(), pt.max())
    ax.set_xlabel("pseudotime")
    ax.set_ylabel("W per cell")
    ax.set_title(f"{gene}: W(t) (Spearman rho={rho_pt:.3f})", fontsize=7)


def plot_gene_4panel(gene, lineage_name, lineage_tag, suffix=""):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
    panel_stacked_bar(axes[0, 0], gene, lineage_name, lineage_tag)
    panel_W_dot(axes[0, 1], gene, lineage_name, lineage_tag)
    panel_smoothed_stack(axes[1, 0], gene, lineage_name, lineage_tag)
    panel_W_trajectory(axes[1, 1], gene, lineage_name, lineage_tag)
    fig.suptitle(f"{gene} ({lineage_name}{suffix})", fontsize=8)
    plt.tight_layout()
    pdf = PANEL_DIR / f"{lineage_tag}_{gene}{suffix}_2x2.pdf"
    png = PANEL_DIR / f"{lineage_tag}_{gene}{suffix}_2x2.png"
    fig.savefig(pdf, bbox_inches="tight", dpi=300)
    fig.savefig(png, bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig)
    return pdf


# --- Selection: 1 ASD gene per (lineage, trajectory_pattern), prefer:
#       (a) not in main_hits top-15 already, then
#       (b) lowest LRT padj.
#     If the only ASD hit for a (lineage, trajectory_pattern) cell is already
#     in top-15, plot it anyway (the requirement explicitly allows this).
selection_rows = []
for ltag in LINEAGE_TAG.values():
    df = kn[ltag]
    sub = df[df["is_asd"] & (df["padj"] < PADJ_THR)].copy()
    if len(sub) == 0:
        continue
    sub["_already_in_top15"] = sub["gene_name"].isin(TOP15_BY_LINEAGE[ltag])
    for traj, group in sub.groupby("trajectory_pattern", observed=True):
        # Sort: not already plotted first, then by padj ascending
        ordered = group.sort_values(
            ["_already_in_top15", "padj"], ascending=[True, True]
        )
        chosen = ordered.iloc[0]
        selection_rows.append(
            {
                "lineage": ltag,
                "trajectory_pattern": traj,
                "gene_name": chosen["gene_name"],
                "padj": chosen["padj"],
                "delta_psi": chosen["delta_psi"],
                "rho_W": chosen["rho_W"],
                "is_li2023": bool(chosen["is_li2023"]),
                "is_sfari_high": bool(chosen["is_sfari_high"]),
                "sfari_score": chosen["sfari_score"],
                "duplicate_of_top15": bool(chosen["_already_in_top15"]),
                "n_asd_in_cell": len(group),
            }
        )

selection_df = pd.DataFrame(selection_rows)
selection_df.to_csv(KN_OUT_DIR / "asd_panel_selection.csv", index=False)
print("\n=== Selection (one ASD gene per [lineage x trajectory_pattern]) ===")
print(selection_df.to_string(index=False))

print("\n=== Plotting panels ===")
plotted = []
for _, row in selection_df.iterrows():
    suffix = "_dup_with_top15" if row["duplicate_of_top15"] else ""
    out = plot_gene_4panel(
        row["gene_name"],
        next(l for l, t in LINEAGE_TAG.items() if t == row["lineage"]),
        row["lineage"],
        suffix=suffix,
    )
    plotted.append(out)
    print(f"  saved {out.name}")

print(f"\nTotal panels: {len(plotted)}")

# TODO:

Looking at the panel functions in the cell above, I can diagnose the issue without needing to run anything.

The root cause is a **data coverage mismatch** between the three failing panels and the one working panel:

**Working panel — `panel_smoothed_stack` (bottom-left)**
Reads from `EFFECT_SIZES_DIR / ltag / "smoothed_proportions.h5ad"` — an AnnData object that was computed for **all genes** during the effect-size calculation pipeline.

**Failing panels — `panel_stacked_bar`, `panel_W_dot`, `panel_W_trajectory`**
All three read from `RESULTS_DIR = KN_BASE / "utrcalling/example_genes_results"`:

- `pseudobulk_proportions.csv`
- `W_per_cluster.csv`
- `W_per_cell.csv`

The directory name `example_genes_results` is the tell — these CSVs were generated in notebook 5 specifically for the **top-N "example" genes** (the main hits by composite score). The ASD genes selected here are a different set, and most of them won't be in those CSVs. Each failing function has the same pattern: when the gene is absent it hits `len(...) == 0` and calls `ax.axis("off")`, silently blanking the panel.

You can confirm by checking:

```python
print(df_W_cluster["gene_name"].unique())  # or df_props, df_W_cell
# vs
print(selection_df["gene_name"].tolist())
```

**Fix plan (two options):**

1. **Quick fix (preferred if notebook 5 is flexible):** Re-run the notebook 5 pipeline with the ASD selection genes added to the input gene list, so the three CSVs get populated with their data too. Then this notebook works as-is.

2. **Self-contained fix (compute on the fly):** Replace the CSV lookups in the three failing panel functions with direct computation from the AnnData objects in `smoothed[lineage_tag]` — the h5ad already has per-cell proportions (`X`), pseudotime (`obs`), and cell-type labels. `W` per cell can be computed as the weighted mean UTR size normalized to [0,1], and per-cluster W can be grouped from that. This makes the ASD panel notebook fully independent of notebook 5's pre-computed files.

Option 2 is cleaner architecturally since this notebook already has everything it needs in the h5ad objects. The `panel_smoothed_stack` function already shows the pattern — you can derive pseudobulk proportions and W values by grouping cells by `celltype` from `sub.obs`.


## Summary


In [ ]:
print("Files in figures/asd/:")
for f in sorted(KN_FIG_DIR.iterdir()):
    print(f"  {f.name}: {f.stat().st_size/1024:.0f} KB")
print("\nFiles in asd_results/:")
for f in sorted(KN_OUT_DIR.iterdir()):
    print(f"  {f.name}: {f.stat().st_size/1024:.0f} KB")